In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from sparsemax import Sparsemax


In [3]:
documents ={
    0: "The capital of France is Paris",
    1: "Python is a popular programming language",
    2: "The Earth revolves around the Sun",
    3: "Neural networks are used in deep learning",
    4: "Tokyo is the capital of Japan"
}

training_data = [
    ("What is the capital of France", 0),
    ("Which city is the capital of France", 0),
    ("Tell me about Paris", 0),

    ("What is Python", 1),
    ("Which programming language is popular", 1),
    ("Tell me about Python programming", 1),

    ("What does the Earth revolve around", 2),
    ("Which planet orbits the Sun", 2),

    ("What are neural networks used for", 3),
    ("Explain deep learning networks", 3),

    ("What is the capital of Japan", 4),
    ("Which city is Japan's capital", 4),
]
validation_data = [
    ("Is Paris the capital of France", 0),
    ("Tell me about the capital of France", 0),
    ("What is a popular programming language", 1),
    ("Tell me about the Python language", 1),
    ("What revolves around the Sun", 2),
    ("Tell me about the Earth", 2),
    ("What is deep learning", 3),
    ("Tell me about neural networks", 3),
    ("Is Tokyo the capital of Japan", 4),
    ("Which city is the capital of Japan", 4),
]


### embedding + MLP

In [4]:
def tokenize(text):  #basic whitespace tokenizer
    return text.lower().replace("?","").split()

vocabulary =set()

for query, _ in training_data:
    vocabulary.update(tokenize(query))

for document in documents.values():
    vocabulary.update(tokenize(document))

word2idx = { word: i+1 for i,word in enumerate(sorted(vocabulary)) }

vocab_sz = len(vocabulary) + 1


In [5]:
def encode(text):
    return torch.tensor( [word2idx[w] for w in tokenize(text)], dtype=torch.long)

- Embedding layer
- Mean pooling to create a query representation
- MLP with one hidden layer

In [17]:
class DSI(nn.Module):
    def __init__(self, vocab_sz, n_embed, n_hidden, n_docs):
        super().__init__()

        self.embedding = nn.Embedding( vocab_sz, n_embed)
        self.fc1 = nn.Linear(n_embed, n_hidden)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(n_hidden, n_docs)

    def forward(self, query):
        x = self.embedding(query)
        #x = x.mean(dim=1)
        mask = (query !=0).unsqueeze(-1)    #filter out padding tokens
        x = (x * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
        
        return self.fc2(self.relu(self.fc1(x)))

In [44]:
model = DSI(vocab_sz, 32, 64, len(documents))
criterion= nn.CrossEntropyLoss()

In [14]:
[encode(q) for q, _ in training_data]

[tensor([36, 14, 33,  5, 22, 12]),
 tensor([37,  6, 14, 33,  5, 22, 12]),
 tensor([32, 19,  2, 24]),
 tensor([36, 14, 28]),
 tensor([37, 27, 17, 14, 26]),
 tensor([32, 19,  2, 28, 27]),
 tensor([36,  8, 33,  9, 29,  4]),
 tensor([37, 25, 23, 33, 31]),
 tensor([36,  3, 21, 20, 35, 11]),
 tensor([10,  7, 18, 20]),
 tensor([36, 14, 33,  5, 22, 15]),
 tensor([37,  6, 14, 16,  5])]

In [46]:
from torch.nn.utils.rnn import pad_sequence

train_X = pad_sequence(
    [encode(query) for query, _ in training_data],
    batch_first = True,
    padding_value=0
)
train_y = torch.tensor([doc_id for _, doc_id in training_data])

valid_X =pad_sequence(
    [encode(query) for query, _ in validation_data],
    batch_first = True,
    padding_value=0
)
valid_y = torch.tensor([doc_id for _, doc_id in validation_data])

In [ ]:
import importlib
import train_harness
importlib.reload(train_harness)
#harness.Trainer?

In [ ]:
learn = train_harness.Trainer(model, (train_X, train_y), (valid_X,valid_y), criterion=criterion,checkpoint_path=None, batch_size=8)

In [ ]:
learn.train_one_cycle(n_epochs=40, max_lr=1e-2, early_stopping=True)

In [47]:
@torch.no_grad()
def search(query):
    model.eval()
    tokens = encode(query).unsqueeze(0).to(learn.device)
    scores = model(tokens)
    doc_id = scores.argmax(dim=1).item()
    return documents[doc_id]

In [48]:
search("Which city is the capital of France?")

'The capital of France is Paris'

### Transformer

In [21]:
PAD, UNK = 0,1

word2idx = { word: i+2 for i, word in enumerate(sorted(vocabulary)) }
vocab_sz = len(vocabulary) + 2

def encode(text):
    tokens = [ word2idx.get(w, UNK) for w in tokenize(text) ]
    return torch.tensor(tokens, dtype=torch.long)

In [12]:
TGT_PAD=0
BOS =1
EOS= 2
BIT_0=3
BIT_1 =4
target_vocab_sz = 5

def encode_doc_id(doc_id):
    binary = format(doc_id + 1, "03b")
    return [ BIT_0 if bit == "0" else BIT_1 for bit in binary ]

encode_doc_id(10)

[4, 3, 4, 4]

In [24]:
encoder_inputs, decoder_inputs, decoder_targets = [], [], []

for query, doc_id  in training_data:
    source = encode(query)
    target = encode_doc_id(doc_id)
    decoder_input = [BOS] + target
    decoder_target = target + [EOS]
    encoder_inputs.append(source)
    decoder_inputs.append( torch.tensor(decoder_input))
    decoder_targets.append(torch.tensor(decoder_target))

X = torch.nn.utils.rnn.pad_sequence( encoder_inputs, batch_first=True, padding_value=PAD)

T = torch.stack(decoder_inputs)
Y = torch.stack(decoder_targets)

In [34]:
class PositionalEmbedding(nn.Module):
    def __init__(self, max_len, n_embed):
        super().__init__()
        self.embedding = nn.Embedding(max_len, n_embed)

    def forward(self, x):
        positions = torch.arange(x.size(1), device = x.device)

        return x + self.embedding(positions)

In [41]:
class TransformerDSI(nn.Module):
    def __init__(self, src_vocab_sz, tgt_vocab_sz, n_embed=64, nhead=4,
                 num_encoder_layers = 2, num_decoder_layers=2,
                 n_feedforward =128, dropout =0.1):
        super().__init__()

        self.d_model = n_embed

        self.src_embedding = nn.Embedding( src_vocab_sz, n_embed, padding_idx=PAD)
        self.src_position = PositionalEmbedding(64, n_embed)
        self.tgt_embedding = nn.Embedding(tgt_vocab_sz, n_embed, padding_idx=TGT_PAD)
        self.tgt_position = PositionalEmbedding(16, n_embed)

        self.transformer = nn.Transformer(
            d_model = n_embed, nhead = nhead, num_encoder_layers = num_encoder_layers,
            num_decoder_layers = num_decoder_layers, dim_feedforward = n_feedforward, dropout=dropout,
            batch_first = True
        )

        self.output_layer = nn.Linear(n_embed, tgt_vocab_sz)


    def forward(self, src, tgt):
        src_padding_mask = src == PAD
        tgt_padding_mask = tgt == TGT_PAD

        src_emb = self.src_embedding(src) * (self.d_model ** 0.5)
        src_emb = self.src_position(src_emb)

        tgt_emb = self.tgt_embedding(tgt) * (self.d_model ** 0.5)
        tgt_emb = self.tgt_position(tgt_emb)

        #Causal mask prevents looking at future token
        tgt_mask = nn.Transformer.generate_square_subsequent_mask( tgt.size(1), device=tgt.device)

        output = self.transformer( src_emb, tgt_emb, tgt_mask=tgt_mask, src_key_padding_mask = src_padding_mask,
                                   tgt_key_padding_mask = tgt_padding_mask,
                                   memory_key_padding_mask = src_padding_mask
                                 )
        return self.output_layer(output)
                 

In [49]:
model = TransformerDSI( src_vocab_sz = vocab_sz, tgt_vocab_sz=target_vocab_sz)
optimizer = torch.optim.Adam(model.parameters(), lr = 0.001)
criterion = nn.CrossEntropyLoss()

In [50]:
#train harness can't handle transformers for now
epochs = 500
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    
    logits = model(X,T)
    loss = criterion( logits.reshape(-1, target_vocab_sz), Y.reshape(-1))
    loss.backward()
    optimizer.step()

    if epoch % 50 == 0:
        print(f'Epoch {epoch}, Loss: {loss.item():.4f}')

/Users/samuelo/.venv/lib/python3.14/site-packages/torch/nn/modules/activation.py:1337: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = F._canonical_mask(


Epoch 0, Loss: 1.7277
Epoch 50, Loss: 0.1521
Epoch 100, Loss: 0.1116
Epoch 150, Loss: 0.0498
Epoch 200, Loss: 0.0335
Epoch 250, Loss: 0.0044
Epoch 300, Loss: 0.0069
Epoch 350, Loss: 0.0022
Epoch 400, Loss: 0.0050
Epoch 450, Loss: 0.0014


In [51]:
def generate_doc_id(model, query):
    model.eval()
    with torch.inference_mode():
        src = encode(query).unsqueeze(0)

        generated = [BOS]

        for _ in range(4):
            tgt = torch.tensor([generated],  dtype = torch.long)

            logits = model(src, tgt)
            next_token = logits[:, -1, :].argmax(dim=-1).item()

            if next_token == EOS:
                break

            generated.append(next_token)

        bits = []
        for token in generated[1:]:
            if token == BIT_0:
                bits.append("0")
            elif token == BIT_1:
                bits.append("1")
            else:
                raise ValueError("Invalid document ID token")

        if len(bits) != 3: raise ValueError("Invalid document ID length")

        doc_id = int("".join(bits), 2) -1
        return doc_id
    

In [52]:
def search(query):
    doc_id = generate_doc_id(model, query)
    if doc_id not in documents:
        return "No matching document found"

    return documents[doc_id]

In [53]:
search("Which is the capital city of Japan")

/Users/samuelo/.venv/lib/python3.14/site-packages/torch/nn/modules/transformer.py:533: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /Users/runner/work/pytorch/pytorch/aten/src/ATen/NestedTensorImpl.cpp:181.)
  output = torch._nested_tensor_from_mask(


'Tokyo is the capital of Japan'